## Here we are we cleaning the dataset. Some of the cleaning is already done in excel.

### I found that the dataset has negative values. Seeing the nature of it, it seems like refunds. So i am going to delete that rows. of negative and posiive values.

In [2]:
import pandas as pd
from collections import defaultdict, deque

# Load your cleaned dataset
file_path = "eden_datasets/UL EDEN Transaction 01.04.25 to 31.03.26.csv"

df = pd.read_csv(file_path)

# Convert TransDate to datetime
df["TransDate"] = pd.to_datetime(df["TransDate"], dayfirst=True, errors="coerce")

# Remove rows with invalid dates, if any
df = df.dropna(subset=["TransDate"]).copy()

# Sort by date so refunds are matched with earlier sales
df = df.sort_values("TransDate").reset_index(drop=True)

# Create a row ID so we can track which rows to delete
df["RowID"] = df.index

# Round money values to avoid decimal precision issues
df["TransValueRounded"] = df["TransValue"].round(2)
df["AbsTransValue"] = df["TransValueRounded"].abs()

# This set will store rows to remove
rows_to_remove = set()

# Store earlier positive sales
# Key = (PLUCode, price)
positive_sales = defaultdict(deque)

# First pass: match refunds to earlier positive sales
for idx, row in df.iterrows():
    value = row["TransValueRounded"]
    plu_code = row["PLUCode"]
    abs_value = row["AbsTransValue"]
    receipt = row["RECEIPT"]
    
    key = (plu_code, abs_value)
    
    if value > 0:
        # Store positive sale row
        positive_sales[key].append(idx)
    
    elif value < 0:
        # This is a refund row
        matched_sale_idx = None
        
        # Prefer matching with same receipt first
        for sale_idx in list(positive_sales[key]):
            if df.loc[sale_idx, "RECEIPT"] == receipt:
                matched_sale_idx = sale_idx
                break
        
        # If no same-receipt match, match nearest earlier sale with same PLUCode and price
        if matched_sale_idx is None and len(positive_sales[key]) > 0:
            matched_sale_idx = positive_sales[key][0]
        
        # If a match is found, remove both sale and refund
        if matched_sale_idx is not None:
            rows_to_remove.add(matched_sale_idx)
            rows_to_remove.add(idx)
            
            # Remove matched sale from available positive sales
            positive_sales[key].remove(matched_sale_idx)

# Create refund-corrected dataset
df_refund_corrected = df[~df.index.isin(rows_to_remove)].copy()

# Remove helper columns before saving
df_refund_corrected = df_refund_corrected.drop(
    columns=["RowID", "TransValueRounded", "AbsTransValue"]
)

# Save new dataset
output_file = "UL_EDEN_refund_corrected_transactions.csv"
df_refund_corrected.to_csv(output_file, index=False)

print("Refund correction completed.")
print("Original rows:", len(df))
print("Rows removed:", len(rows_to_remove))
print("Final rows:", len(df_refund_corrected))
print("Saved file as:", output_file)

/var/folders/61/pw_dwqt140ndz64pvx21l9040000gn/T/ipykernel_860/3827497064.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["TransDate"] = pd.to_datetime(df["TransDate"], dayfirst=True, errors="coerce")


Refund correction completed.
Original rows: 154978
Rows removed: 88
Final rows: 154890
Saved file as: UL_EDEN_refund_corrected_transactions.csv


### Now i am just auditing the done code

In [3]:
# Save removed refund-pair rows for checking/audit
refund_pair_audit = df[df.index.isin(rows_to_remove)].copy()

refund_pair_audit = refund_pair_audit.drop(
    columns=["RowID", "TransValueRounded", "AbsTransValue"]
)

audit_file = "UL_EDEN_removed_refund_pairs_audit.csv"
refund_pair_audit.to_csv(audit_file, index=False)

print("Audit file saved as:", audit_file)
print("Rows in audit file:", len(refund_pair_audit))

Audit file saved as: UL_EDEN_removed_refund_pairs_audit.csv
Rows in audit file: 88


In [4]:
df_check = pd.read_csv("UL_EDEN_refund_corrected_transactions.csv")

negative_rows_left = df_check[df_check["TransValue"] < 0]

print("Negative rows still left:", len(negative_rows_left))

negative_rows_left.head()

Negative rows still left: 810


,RECEIPT,TransDate,TransValue,PLUName,GroupCode,GroupName,PLUCode
72616,362234,2025-10-14 10:18:00,-0.15,OWN KEEP CUP -0.15,1,HOT BEVS,4262801
72617,362234,2025-10-14 10:18:00,-0.15,OWN KEEP CUP -0.15,1,HOT BEVS,4262801
73215,362543,2025-10-14 13:36:00,-0.15,OWN KEEP CUP -0.15,1,HOT BEVS,4262801
73249,362563,2025-10-14 13:52:00,-0.15,OWN KEEP CUP -0.15,1,HOT BEVS,4262801
73400,362653,2025-10-14 15:00:00,-0.15,OWN KEEP CUP -0.15,1,HOT BEVS,4262801


### foud that there is a transaction -ve that is called own keep cup. Have asled the data proider for clarification but until then I am assuming it is the customer using their own cup so dicount is given for that and that is not needed for forcasting order. Unless there is change in the scope to reduce economical expenses.

In [6]:
import pandas as pd

df = pd.read_csv("eden_datasets/UL_EDEN_refund_corrected_transactions.csv")

# Separate OWN KEEP CUP rows for audit
own_cup_rows = df[df["PLUName"].str.contains("OWN KEEP CUP", case=False, na=False)].copy()

# Remove OWN KEEP CUP rows from final demand transaction dataset
df_final = df[~df["PLUName"].str.contains("OWN KEEP CUP", case=False, na=False)].copy()

# Save own cup rows separately
own_cup_rows.to_csv("UL_EDEN_own_cup_discount_rows_audit.csv", index=False)

# Save final cleaned transaction file
df_final.to_csv("UL_EDEN_final_clean_transactions.csv", index=False)

print("Original rows:", len(df))
print("OWN KEEP CUP rows removed:", len(own_cup_rows))
print("Final rows:", len(df_final))
print("Negative values remaining:", len(df_final[df_final["TransValue"] < 0]))

Original rows: 154890
OWN KEEP CUP rows removed: 810
Final rows: 154080
Negative values remaining: 0


### We need to check if there is any blnk missing values in the code

In [8]:
df = pd.read_csv("eden_datasets/UL_EDEN_final_clean_transactions.csv")

# Check missing values in all columns
print(df.isnull().sum())

RECEIPT       0
TransDate     0
TransValue    0
PLUName       0
GroupCode     0
GroupName     0
PLUCode       0
dtype: int64


In [9]:
important_columns = ["TransDate", "TransValue", "PLUName", "GroupName", "PLUCode"]

print(df[important_columns].isnull().sum())

TransDate     0
TransValue    0
PLUName       0
GroupName     0
PLUCode       0
dtype: int64


In [10]:
for col in important_columns:
    blank_count = df[col].astype(str).str.strip().eq("").sum()
    print(col, "blank values:", blank_count)

TransDate blank values: 0
TransValue blank values: 0
PLUName blank values: 0
GroupName blank values: 0
PLUCode blank values: 0


### we need to change the TransDate to more featues like time, day of the week, mnth etc

In [12]:
df = pd.read_csv("eden_datasets/UL_EDEN_final_clean_transactions.csv")

# Convert TransDate from text to datetime
df["TransDate"] = pd.to_datetime(df["TransDate"], dayfirst=True, errors="coerce")

# Check if any dates failed to convert
print("Invalid TransDate rows:", df["TransDate"].isnull().sum())

# Create date/time features
df["Date"] = df["TransDate"].dt.date
df["Hour"] = df["TransDate"].dt.hour
df["DayOfWeek"] = df["TransDate"].dt.day_name()
df["Month"] = df["TransDate"].dt.month
df["WeekOfYear"] = df["TransDate"].dt.isocalendar().week.astype(int)

# Save updated file
df.to_csv("UL_EDEN_final_clean_transactions_with_dates.csv", index=False)

df.head()

/var/folders/61/pw_dwqt140ndz64pvx21l9040000gn/T/ipykernel_860/1677103538.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["TransDate"] = pd.to_datetime(df["TransDate"], dayfirst=True, errors="coerce")


Invalid TransDate rows: 0


,RECEIPT,TransDate,TransValue,PLUName,GroupCode,GroupName,PLUCode,Date,Hour,DayOfWeek,Month,WeekOfYear
0,194814,2025-07-16 14:46:00,1026.0,VEGT MAINS 3,7,DINNER,4241483,2025-07-16,14,Wednesday,7,29
1,194832,2025-07-17 14:14:00,549.0,MAINS 3,7,DINNER,4241480,2025-07-17,14,Thursday,7,29
2,194782,2025-09-07 14:27:00,513.0,MAINS 3,7,DINNER,4241480,2025-09-07,14,Sunday,9,36
3,194837,2025-07-18 14:35:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-18,14,Friday,7,29
4,194801,2025-10-07 14:51:00,468.0,MAINS 3,7,DINNER,4241480,2025-10-07,14,Tuesday,10,41


### in the dataset there is no information of how many units sold. So assuming until the data provider confirm, that eac row is one unit sold. we are adding that feature as well.

In [15]:
import pandas as pd

# Load the dataset with date features
df = pd.read_csv("eden_datasets/UL_EDEN_final_clean_transactions_with_dates.csv")

# Add UnitSold column
# Each row represents one sold unit/item
df["UnitSold"] = 1

# Save as a new dataset
df.to_csv("UL_EDEN_transactions_with_date_features_and_unitsold.csv", index=False)

print("UnitSold column added successfully.")
print("Rows:", len(df))
df.head()

UnitSold column added successfully.
Rows: 154080


,RECEIPT,TransDate,TransValue,PLUName,GroupCode,GroupName,PLUCode,Date,Hour,DayOfWeek,Month,WeekOfYear,UnitSold
0,194814,2025-07-16 14:46:00,1026.0,VEGT MAINS 3,7,DINNER,4241483,2025-07-16,14,Wednesday,7,29,1
1,194832,2025-07-17 14:14:00,549.0,MAINS 3,7,DINNER,4241480,2025-07-17,14,Thursday,7,29,1
2,194782,2025-09-07 14:27:00,513.0,MAINS 3,7,DINNER,4241480,2025-09-07,14,Sunday,9,36,1
3,194837,2025-07-18 14:35:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-18,14,Friday,7,29,1
4,194801,2025-10-07 14:51:00,468.0,MAINS 3,7,DINNER,4241480,2025-10-07,14,Tuesday,10,41,1


In [16]:
df[["TransDate", "PLUName", "TransValue", "UnitSold"]].head(10)

,TransDate,PLUName,TransValue,UnitSold
0,2025-07-16 14:46:00,VEGT MAINS 3,1026.0,1
1,2025-07-17 14:14:00,MAINS 3,549.0,1
2,2025-09-07 14:27:00,MAINS 3,513.0,1
3,2025-07-18 14:35:00,MAINS 3,513.0,1
4,2025-10-07 14:51:00,MAINS 3,468.0,1
5,2025-07-24 12:53:00,VEGT MAINS 3,468.0,1
6,2025-01-08 13:30:00,MAINS 3,450.0,1
7,2025-07-22 13:21:00,MAINS 3,441.0,1
8,2025-07-22 13:21:00,MAINS 3,441.0,1
9,2025-07-31 13:13:00,MAINS 3,432.0,1


### from the data provider I understoof that the own cup is indeed a discount for bringing their own cups. So we dont need that. It will be better to delete that discounts for a cleaner data set

### since that was all done with assumption we are going to the next step that is to delete the rows of DRS 15c which is the can deposit scheme. we dont need that.

In [7]:
import pandas as pd

# Load your current clean dataset
df = pd.read_csv("eden_datasets/UL_EDEN_transactions_with_date_features_and_unitsold.csv")

# Find DRS 15C rows
drs_rows = df[df["PLUName"].str.contains("DRS 15C", case=False, na=False)].copy()

# Remove DRS 15C rows
df_no_drs = df[~df["PLUName"].str.contains("DRS 15C", case=False, na=False)].copy()

# Save removed DRS rows for audit
drs_rows.to_csv("eden_datasets/UL_EDEN_DRS_15C_removed_audit.csv", index=False)

# Save new cleaned dataset
df_no_drs.to_csv("eden_datasets/UL_EDEN_clean_no_negative_no_owncup_no_drs.csv", index=False)

print("DRS 15C removal completed.")
print("Original rows:", len(df))
print("DRS 15C rows removed:", len(drs_rows))
print("Final rows:", len(df_no_drs))
print("Negative values remaining:", len(df_no_drs[df_no_drs["TransValue"] < 0]))

DRS 15C removal completed.
Original rows: 154080
DRS 15C rows removed: 15097
Final rows: 138983
Negative values remaining: 0


In [8]:
drs_rows[["RECEIPT", "TransDate", "TransValue", "PLUName", "GroupName", "PLUCode"]].head(20)

,RECEIPT,TransDate,TransValue,PLUName,GroupName,PLUCode
138952,204075,2025-01-04 10:43:00,0.15,DRS 15C,Confectionary,100000015
138953,204085,2025-01-04 10:49:00,0.15,DRS 15C,Confectionary,100000015
138954,204130,2025-01-04 11:52:00,0.15,DRS 15C,Confectionary,100000015
138955,204130,2025-01-04 11:52:00,0.15,DRS 15C,Confectionary,100000015
138956,204137,2025-01-04 12:02:00,0.15,DRS 15C,Confectionary,100000015
138957,204144,2025-01-04 12:05:00,0.15,DRS 15C,Confectionary,100000015
138958,204146,2025-01-04 12:06:00,0.15,DRS 15C,Confectionary,100000015
138959,204153,2025-01-04 12:07:00,0.15,DRS 15C,Confectionary,100000015
138960,204158,2025-01-04 12:12:00,0.15,DRS 15C,Confectionary,100000015
138961,204162,2025-01-04 12:13:00,0.15,DRS 15C,Confectionary,100000015


In [9]:
df_check = pd.read_csv("eden_datasets/UL_EDEN_clean_no_negative_no_owncup_no_drs.csv")

print("DRS 15C rows left:", len(
    df_check[df_check["PLUName"].str.contains("DRS 15C", case=False, na=False)]
))

DRS 15C rows left: 0


### OPEN UL was confirmed by the data provoder as a miscellanous purchase by the unversity of limerick for special orders and thre for we dont need it in ingredient mappin but it is better to put it in demand forcasting prediction tho. But the ingredient mapping can be done by the restaurant based on the forecast prediction we give. So now we are just adding a 